# Q8.
```{admonition}
:class: note
Pick $10$ images of animals. Use a pretrained image classification CNN to predict the class of each of your images, and report the probabilities for the top five predicted classes for each image.

In [2]:
import numpy as np
import pandas as pd
from glob import glob
import json

In [3]:
import torch
from torchvision.io import read_image
from torchvision.models import resnet50,ResNet50_Weights
from torchvision.transforms import Resize,Normalize,CenterCrop

In [4]:
#Images pulled from San Diego Zoo website
imgfiles = [f for f in glob('img/*')]

In [5]:
resize = Resize((232,232))
crop = CenterCrop(224)
normalize = Normalize([0.485 ,0.456 ,0.406],
                      [0.229 ,0.224 ,0.225])

In [6]:
imgs = normalize(torch.stack([torch.div(crop(resize(read_image(f))), 255) for f in imgfiles]))

In [7]:
with open('imagenet_class_index.json') as f:
    labels = json.load(f)

class_labels = pd.DataFrame(
    [(int(k),v[1]) for k,v in labels.items()],
    columns = ['idx','label']
).set_index('idx').sort_index()

In [8]:
resnet_model = resnet50(weights=ResNet50_Weights.DEFAULT)
resnet_model.eval();

In [9]:
img_preds = resnet_model(imgs)
img_probs = torch.nn.functional.softmax(resnet_model(imgs),dim=1).detach().numpy()

In [10]:
for i, imgfile in enumerate(imgfiles):
    img_df = class_labels.copy()
    img_df['prob'] = img_probs[i]
    img_df = img_df.sort_values(by='prob', ascending=False )[:5]
    print(f'Image: {imgfile}')
    print(img_df.reset_index().drop(columns =['idx']))
    print('')

Image: img\animals_hero_americankestrel.jpg
            label      prob
0       brambling  0.079351
1             jay  0.044677
2            kite  0.037543
3  indigo_bunting  0.022284
4           quail  0.010247

Image: img\animals_hero_ants.jpg
         label      prob
0          ant  0.221434
1      cricket  0.020135
2   leafhopper  0.008249
3       mantis  0.006673
4  grasshopper  0.006112

Image: img\animals_hero_armadillo_0.jpg
             label      prob
0        armadillo  0.443106
1  prairie_chicken  0.002194
2             hare  0.001523
3          wallaby  0.001512
4      wood_rabbit  0.001391

Image: img\animals_hero_bald_eagle.jpg
          label      prob
0    bald_eagle  0.370977
1       vulture  0.008323
2          kite  0.006094
3  black_grouse  0.001930
4   black_stork  0.001471

Image: img\animals_hero_caracal.jpg
          label      prob
0          lynx  0.544346
1        cougar  0.007781
2        impala  0.005143
3  Egyptian_cat  0.003795
4          hare  0.003194
